<a href="https://colab.research.google.com/github/ammar-aa/Fly_rank_internship_repo/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
from google.colab import userdata
auth=userdata.get("HF_TOKEN")
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
con=duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{auth}'
);
""")
from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
df = con.sql(f"""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
dfF = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
dfM = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()

In [5]:
df['ctr'] = df['gsc_clicks'] / df['gsc_impressions']
df['avg_engagement_sec_per_session'] = df['ga4_total_engagement_sec'] / df['ga4_sessions']
df['scroll_rate'] = df['scroll_events'] / df['ga4_pageviews']

for col in ['ctr', 'avg_engagement_sec_per_session', 'scroll_rate']:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    df[col] = df[col].astype('float64')

print(df['ctr'].dtype)
print(df['avg_engagement_sec_per_session'].dtype)
print(df['scroll_rate'].dtype)

float64
float64
float64


In [6]:
df_trend = dfM.merge(dfF, on=['client_hash_id', 'content_hash_id'], suffixes=('_feb', '_mar'), how='outer')

In [7]:
df_trend = df_trend[df_trend['gsc_impressions_feb'] >= 30]
df_trend = df_trend[df_trend['gsc_impressions_mar'] > 0]

df_trend['trend_pct'] = (
    (df_trend['gsc_impressions_mar'] - df_trend['gsc_impressions_feb'])
    / df_trend['gsc_impressions_feb']
) * 100

clip_value = df_trend['trend_pct'].quantile(0.99)
df_trend['trend_pct'] = df_trend['trend_pct'].clip(lower=-clip_value, upper=clip_value)

In [8]:
df['ga4_data_available'] = df['ga4_data_available'].fillna(False)

conditions = [
    df['gsc_data_available'] & df['ga4_data_available'],
    df['gsc_data_available'] & ~df['ga4_data_available'],
    ~df['gsc_data_available'] & df['ga4_data_available'],
    ~df['gsc_data_available'] & ~df['ga4_data_available']
]

tiers = ['Full', 'GSC-only', 'GA4-only', 'unavailable']

df['tier'] = np.select(conditions, tiers, default=None)

In [9]:
df = df.groupby(['client_hash_id', 'content_hash_id'], as_index=False).agg(
    gsc_sum_position=('gsc_sum_position', 'sum'),
    gsc_avg_position=('gsc_avg_position', 'mean'),
    gsc_impressions=('gsc_impressions', 'sum'),
    gsc_clicks=('gsc_clicks', 'sum'),
    ctr=('ctr', 'mean'),
    ga4_pageviews=('ga4_pageviews', 'sum'),
    ga4_sessions=('ga4_sessions', 'sum'),
    ga4_users=('ga4_users', 'sum'),
    ga4_engaged_sessions=('ga4_engaged_sessions', 'sum'),
    ga4_total_engagement_sec=('ga4_total_engagement_sec', 'sum'),
    avg_engagement_sec_per_session=('avg_engagement_sec_per_session', 'mean'),
    scroll_events=('scroll_events', 'sum'),
    gsc_data_available=('gsc_data_available', 'first'),
    ga4_data_available=('ga4_data_available', 'first'),
    tier=('tier', 'first'),
    scroll_rate=('scroll_rate', 'mean')
)

In [10]:
df = df.merge(df_trend[['client_hash_id', 'content_hash_id', 'trend_pct']], on=['client_hash_id', 'content_hash_id'], how='left')

In [11]:
df_full_tier = df[df['tier'] == 'Full']
conditions = [
    df_full_tier['trend_pct'] < -50,
    (df_full_tier['trend_pct'] >= -50) & (df_full_tier['trend_pct'] < -15),
    (df_full_tier['trend_pct'] >= -15) & (df_full_tier['trend_pct'] <= 15),
    (df_full_tier['trend_pct'] > 15) & (df_full_tier['trend_pct'] <= 50),
    df_full_tier['trend_pct'] > 50
]
ranks = ['5-Sharp decline', '4-Mild decline', '3-Flat', '2-Mild growth', '1-Strong growth']
df_full_tier['trend_dir'] = np.select(conditions, ranks, default=None)

In [13]:
display(df_full_tier.groupby('trend_dir').agg(
    {
        'gsc_sum_position': ['mean', 'count'],
        'gsc_avg_position': ['mean', 'count'],
        'gsc_impressions' : ['mean', 'count'],
        'gsc_clicks' : ['mean', 'count'],
        'ga4_pageviews' : ['mean', 'count'],
        'ga4_sessions' : ['mean', 'count'],
        'ga4_users' : ['mean', 'count'],
        'ga4_engaged_sessions' : ['mean', 'count'],
        'ctr' : ['mean', 'count'],
        'ga4_total_engagement_sec' : ['mean', 'count'],
        'avg_engagement_sec_per_session' : ['mean', 'count'],
        'scroll_events' : ['mean', 'count'],
        'scroll_rate' : ['mean', 'count']
    }
))

gsc_sum_position       gsc_avg_position       gsc_impressions  \
                            mean count             mean count            mean   
trend_dir                                                                       
1-Strong growth     38631.748031   254        13.353879   254     3045.051181   
2-Mild growth       45638.026525   377        12.536609   377     4357.331565   
3-Flat              88217.990816   980        14.127915   980     6512.854082   
4-Mild decline     184232.610397  1789        18.189271  1789     8499.656792   
5-Sharp decline    223852.726792  1702        17.747662  1702    10695.907756   

                      gsc_clicks       ga4_pageviews        ...       ctr  \
                count       mean count          mean count  ...      mean   
trend_dir                                                   ...             
1-Strong growth   254  14.846457   254     55.625984   254  ...  0.005111   
2-Mild growth     377  26.655172   377     83.517241   377  ...  0.006050   
3-Flat            980  32.988776   980     92.705102   980  ...  0.005645   
4-Mild decline   1789  26.999441  1789     92.787032  1789  ...  0.004818   
5-Sharp decline  1702  35.373090  1702     78.090482  1702  ...  0.005353   

                      ga4_total_engagement_sec        \
                count                     mean count   
trend_dir                                              
1-Strong growth   254               179.007874   254   
2-Mild growth     377               339.525199   377   
3-Flat            980               364.755102   980   
4-Mild decline   1789               335.869201  1789   
5-Sharp decline  1702               340.943596  1702   

                avg_engagement_sec_per_session       scroll_events        \
                                          mean count          mean count   
trend_dir                                                                  
1-Strong growth                       5.294051   254      7.488189   254   
2-Mild growth                         8.061824   377       9.29443   377   
3-Flat                                7.260497   980     10.503061   980   
4-Mild decline                        7.073323  1788     11.055897  1789   
5-Sharp decline                       5.838988  1698     12.713866  1702   

                scroll_rate        
                       mean count  
trend_dir                          
1-Strong growth    0.146207   254  
2-Mild growth      0.138781   377  
3-Flat             0.134936   980  
4-Mild decline     0.138958  1789  
5-Sharp decline    0.197115  1702  

[5 rows x 26 columns]

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
def build_tier_frames(df):
    df_full = df[df['tier'] == 'Full'].copy()
    df_gsc_only = df[df['tier'] == 'GSC-only'].copy()
    df_ga4_only = df[df['tier'] == 'GA4-only'].copy()

    df_full = df_full.dropna(subset=['trend_pct'])
    df_gsc_only = df_gsc_only.dropna(subset=['trend_pct'])

    df_full['ga4_sessions_pct_rank'] = df_full.groupby('client_hash_id')['ga4_sessions'].transform(
        lambda x: x.rank(pct=True)
    )
    df_full['needs_refresh'] = (
        (df_full['trend_pct'] < -15) &
        (df_full['ga4_sessions_pct_rank'] <= 0.15)
    ).astype(int)

    df_gsc_only['needs_refresh'] = (
        (df_gsc_only['trend_pct'] < -15) &
        (df_gsc_only['gsc_avg_position'] > 10)
    ).astype(int)

    df_ga4_only['needs_refresh'] = (
        (df_ga4_only['ga4_sessions'] <= 2) &
        (df_ga4_only['ga4_engaged_sessions'] == 0)
    ).astype(int)

    return df_full, df_gsc_only, df_ga4_only

In [15]:
df_full, df_gsc_only, df_ga4_only = build_tier_frames(df)

In [16]:
display(df_full['needs_refresh'].value_counts())
display(df_gsc_only['needs_refresh'].value_counts())
display(df_ga4_only['needs_refresh'].value_counts())

,count
needs_refresh,
0,4567
1,535


,count
needs_refresh,
0,63063
1,27196


,count
needs_refresh,
1,603
0,404


In [17]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

norm_avg_position = normalize(df_full['gsc_avg_position'])
norm_ctr = normalize(df_full['ctr'])
norm_clicks = normalize(df_full['gsc_clicks'])
norm_engaged_sessions = normalize(df_full['ga4_engaged_sessions'])
norm_scroll = normalize(df_full['scroll_rate'])
norm_engagement_sec = normalize(df_full['avg_engagement_sec_per_session'])
norm_pv_users_ratio = normalize(df_full['ga4_pageviews'] / df_full['ga4_users'])

df_full['score'] = (

     0.25 * norm_avg_position +
    -0.15 * norm_ctr +
     0.10 * norm_clicks +

    -0.117 * norm_engaged_sessions +
    -0.117 * norm_scroll +
    -0.117 * norm_engagement_sec +

    -0.15 * norm_pv_users_ratio
)

In [18]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

norm_sum_position = normalize(df_gsc_only['gsc_sum_position'])
norm_ctr = normalize(df_gsc_only['ctr'])
norm_clicks = normalize(df_gsc_only['gsc_clicks'])

df_gsc_only['score'] = (
     0.30 * norm_sum_position +
    -0.50 * norm_ctr +
     0.20 * norm_clicks
)

In [19]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

norm_engaged_sessions = normalize(df_ga4_only['ga4_engaged_sessions'])
norm_users = normalize(df_ga4_only['ga4_users'])
norm_engagement_sec = normalize(df_ga4_only['avg_engagement_sec_per_session'])
norm_pageviews = normalize(df_ga4_only['ga4_pageviews'])
norm_scroll = normalize(df_ga4_only['scroll_rate'])

df_ga4_only['score'] = (
    -0.40 * norm_engaged_sessions +
    -0.20 * norm_users +
    -0.15 * norm_engagement_sec +
    -0.15 * norm_pageviews +
    -0.10 * norm_scroll
)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.